# B3 assembled+direct (cuDSS) solver — advisor follow-up review, point 7

**EXPERIMENTAL, NOT YET RUN ON GPU, NOT A FINALIZED RESULT** — see `b3_assembled_direct_solver.py`'s own module docstring for the full detail. Short version: this ports this study's own GPU-native assembled+direct approach (explicit sparse tangent, cuDSS direct solve, analysis-reuse + symmetric storage) — already measured for B1's 2D mesh — to B3's 3D hex mesh, per the advisor's explicit request ("The optimized assembled/direct solver MUST be included ... For B3, also benchmark this solver at relevant resolutions if possible").

**What has already been verified (CPU, no GPU needed, done before this notebook was written):** the sparse-tangent assembly was checked against a dense `torch.autograd` Hessian of the exact same energy at three small meshes (machine-precision agreement), and a hand-rolled CPU Newton+dense-solve loop using this module's own residual/Jacobian was checked end-to-end against `data/mesh_convergence_B3.py`'s own already-trusted torch-fem reference solve at three small meshes — also machine-precision agreement, after two real bugs were found and fixed this way (a node-dof ordering mismatch, a `vmap`+`hessian`+`slogdet` composition bug, and a boolean-mask-vs-index-array bug in the Dirichlet-BC construction; see the module docstring and `run_correctness_check`'s own docstring for the full account, including one case where the check correctly caught a GEOMETRY mismatch — comparing against the wrong groove depth — rather than a real solver bug).

**What this notebook verifies that CPU testing could not:** the actual `reuse_analysis=True` cuDSS path (`_newton_cudss_reuse_analysis`, CUDA-only — cuDSS has no CPU path) at a real resolution. **Cell 1 below reruns the correctness check on THIS GPU before any timing number is produced** — `run_b3_benchmark` refuses to run otherwise. Read its PASS/FAIL output before trusting anything from Cell 2.

**Do not report, quote, or put into the paper any number from Cell 2 until**: (a) Cell 1 printed PASS, and (b) this has been shown to the advisor, per this project's own standing policy for every solver change of this kind.

In [ ]:
# =====================================================================
#  CELL 1 — setup + MANDATORY correctness gate. Must print PASS before
#  Cell 2 (the actual benchmark) is allowed to run.
# =====================================================================
import os, subprocess, sys

def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)

from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
                    'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

import torch
assert torch.cuda.is_available(), (
    'This notebook needs a CUDA GPU -- the reuse_analysis=True cuDSS path has no '
    'CPU fallback. Runtime > Change runtime type > GPU.')
print('GPU:', torch.cuda.get_device_name(0))

# torch_sla is assembled_direct_solver.py's own existing dependency (already
# required for the 2D assembled+direct study) -- install it the same way that
# study's own notebooks do if it isn't already present in this runtime.
try:
    import torch_sla  # noqa: F401
except ImportError:
    raise RuntimeError(
        'torch_sla is not installed in this runtime. Install it the same way '
        "this project's existing 2D assembled+direct-solver notebooks do "
        '(see Round6_Assembled_Direct_Reuse_Analysis.ipynb / '
        'Round6_Assembled_Direct_Speedup_Production.ipynb for the exact install '
        'cell), then re-run this cell.')

from omar_pfem.b3_assembled_direct_solver import run_correctness_check

print('\n--- Running the mandatory correctness gate (small mesh, real GPU) ---')
max_abs, max_rel, passed = run_correctness_check(Ntheta=9, Nr=7, Nz=6)
print(f"\n{'PASS' if passed else 'FAIL'}: max_abs={max_abs:.3e}, max_rel={max_rel:.3e}")
print('If this printed PASS, Cell 2 is safe to run. If it raised, STOP -- do not')
print('run Cell 2 or trust any output from it until this is root-caused.')

In [ ]:
# =====================================================================
#  CELL 2 — the actual benchmark. Refuses to run unless Cell 1 just
#  passed in this same session (run_b3_benchmark checks this internally).
# =====================================================================
import os, json
from omar_pfem.b3_assembled_direct_solver import run_b3_benchmark, DEFAULT_RESOLUTION

R = '/content/drive/MyDrive/pfem_run'
OUT_JSON = f'{R}/b3_assembled_direct_benchmark.json'
os.makedirs(R, exist_ok=True)

# DEFAULT_RESOLUTION is B3's real 6,840-element production mesh
# (Ntheta=21, Nr=20, Nz=19). Add more (Ntheta,Nr,Nz) triples here from this
# project's own mesh-convergence ladder for an apples-to-apples comparison
# with already-published B3 numbers -- not guessed here, since the exact
# triples behind the ladder's published element counts (480,320 / 839,040 /
# 950,400) are not hardcoded in this module.
RESOLUTIONS = [DEFAULT_RESOLUTION]

rows = run_b3_benchmark(
    RESOLUTIONS, OUT_JSON,
    reuse_analysis=True, matrix_type='general', seed=0)

print('\nResults:')
print(json.dumps(rows, indent=2))
print(f'\nWritten to {OUT_JSON}')
print('Remember: this is still EXPERIMENTAL output -- show it to the advisor')
print('before quoting it anywhere.')